# Demo Notebook: Medical Abstract Sentence Classification

This notebook demonstrates how to load trained models and perform inference on new biomedical text.
We compare the **RNN/LSTM baseline** with the **BioBERT transformer** model.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from src.data_loader import (
    load_pubmed_rct, PubMedRCTDataset, build_vocab, encode_text,
    preprocess_text, ID_TO_LABEL, NUM_LABELS, LABEL_MAP,
)
from src.rnn_model import LSTMClassifier
from src.evaluate import (
    compute_metrics, print_classification_report,
    plot_confusion_matrix, plot_training_curves,
)

RESULTS_DIR = os.path.join('..', 'results')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Load Test Data

In [ ]:
_, _, test_split = load_pubmed_rct()
test_dataset = PubMedRCTDataset(test_split, max_len=128)
print(f'Test set size: {len(test_dataset):,} sentences')

## 2. Load & Evaluate RNN Model

In [ ]:
# Load vocabulary
vocab_path = os.path.join(RESULTS_DIR, 'vocab.json')
with open(vocab_path) as f:
    vocab = json.load(f)
print(f'Vocabulary size: {len(vocab):,}')

# Load RNN model
rnn_model = LSTMClassifier(
    vocab_size=len(vocab), embed_dim=128, hidden_dim=256,
    num_layers=2, num_classes=NUM_LABELS, dropout=0.3,
)
rnn_model.load_state_dict(torch.load(
    os.path.join(RESULTS_DIR, 'best_rnn_model.pt'),
    map_location=DEVICE, weights_only=True
))
rnn_model.to(DEVICE)
rnn_model.eval()
print('RNN model loaded successfully.')

In [ ]:
# Evaluate RNN on test set
rnn_preds = []
rnn_labels = []
MAX_LEN = 128

with torch.no_grad():
    for i in range(0, len(test_dataset), 256):
        batch = [test_dataset[j] for j in range(i, min(i + 256, len(test_dataset)))]
        texts, labels = zip(*batch)
        input_ids = torch.tensor(
            [encode_text(t, vocab, MAX_LEN) for t in texts], dtype=torch.long
        ).to(DEVICE)
        outputs = rnn_model(input_ids)
        preds = outputs.argmax(dim=1).cpu().numpy()
        rnn_preds.extend(preds)
        rnn_labels.extend(labels)

rnn_metrics = compute_metrics(rnn_labels, rnn_preds)
print(f"RNN Test Accuracy: {rnn_metrics['accuracy']:.4f}")
print(f"RNN Test Macro F1: {rnn_metrics['macro_f1']:.4f}")
print()
print_classification_report(rnn_labels, rnn_preds)

In [ ]:
# RNN Confusion Matrix
plot_confusion_matrix(
    rnn_labels, rnn_preds,
    title='RNN/LSTM - Confusion Matrix',
    save_path=os.path.join(RESULTS_DIR, 'rnn_confusion_matrix.png')
)

## 3. Load & Evaluate Transformer Model

In [ ]:
# Load BioBERT model
transformer_dir = os.path.join(RESULTS_DIR, 'best_transformer_model')
tokenizer = AutoTokenizer.from_pretrained(transformer_dir)
transformer_model = AutoModelForSequenceClassification.from_pretrained(transformer_dir)
transformer_model.to(DEVICE)
transformer_model.eval()
print('Transformer model loaded successfully.')

In [ ]:
# Evaluate Transformer on test set
from datasets import load_dataset
from src.transformer_model import tokenize_dataset

dataset = load_dataset('armanc/pubmed-rct20k')
test_tok = tokenize_dataset(dataset['test'], tokenizer, 128)

trans_preds = []
trans_labels = []

with torch.no_grad():
    for i in range(0, len(test_tok), 256):
        batch_ids = test_tok[i:i+256]['input_ids'].to(DEVICE)
        batch_mask = test_tok[i:i+256]['attention_mask'].to(DEVICE)
        outputs = transformer_model(input_ids=batch_ids, attention_mask=batch_mask)
        preds = outputs.logits.argmax(dim=1).cpu().numpy()
        trans_preds.extend(preds)
        trans_labels.extend(test_tok[i:i+256]['labels'].numpy())

trans_metrics = compute_metrics(trans_labels, trans_preds)
print(f"Transformer Test Accuracy: {trans_metrics['accuracy']:.4f}")
print(f"Transformer Test Macro F1: {trans_metrics['macro_f1']:.4f}")
print()
print_classification_report(trans_labels, trans_preds)

In [ ]:
# Transformer Confusion Matrix
plot_confusion_matrix(
    trans_labels, trans_preds,
    title='BioBERT - Confusion Matrix',
    save_path=os.path.join(RESULTS_DIR, 'transformer_confusion_matrix.png')
)

## 4. Model Comparison

In [ ]:
# Side-by-side comparison
import pandas as pd

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1'],
    'RNN/LSTM': [
        rnn_metrics['accuracy'], rnn_metrics['macro_precision'],
        rnn_metrics['macro_recall'], rnn_metrics['macro_f1'],
    ],
    'BioBERT': [
        trans_metrics['accuracy'], trans_metrics['macro_precision'],
        trans_metrics['macro_recall'], trans_metrics['macro_f1'],
    ],
}).round(4)

print(comparison.to_string(index=False))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['RNN/LSTM'], width, label='RNN/LSTM', color='steelblue')
bars2 = ax.bar(x + width/2, comparison['BioBERT'], width, label='BioBERT', color='coral')

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Metric'])
ax.legend()
ax.set_ylim(0, 1.05)
ax.bar_label(bars1, fmt='%.3f', fontsize=8)
ax.bar_label(bars2, fmt='%.3f', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Interactive Inference Demo

In [ ]:
def predict_sentence(text, model_type='transformer'):
    """Predict the section label for a given biomedical sentence."""
    if model_type == 'transformer':
        inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = transformer_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    else:
        processed = preprocess_text(text)
        input_ids = torch.tensor([encode_text(processed, vocab, 128)], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            outputs = rnn_model(input_ids)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]

    pred_label = ID_TO_LABEL[probs.argmax()]
    return pred_label, {ID_TO_LABEL[i]: float(p) for i, p in enumerate(probs)}


# Example biomedical sentences
test_sentences = [
    "Diabetes mellitus is a chronic metabolic disorder affecting millions worldwide.",
    "The aim of this study was to evaluate the efficacy of metformin in type 2 diabetes.",
    "A randomized controlled trial was conducted with 200 patients over 12 months.",
    "Patients treated with metformin showed a significant reduction in HbA1c levels (p<0.001).",
    "These findings suggest that metformin remains a first-line treatment for type 2 diabetes.",
]

expected_labels = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']

print('=' * 80)
print('INFERENCE DEMO')
print('=' * 80)

for sent, expected in zip(test_sentences, expected_labels):
    # Transformer prediction
    t_label, t_probs = predict_sentence(sent, 'transformer')
    # RNN prediction
    r_label, r_probs = predict_sentence(sent, 'rnn')
    
    print(f'\nSentence: "{sent[:80]}..."' if len(sent) > 80 else f'\nSentence: "{sent}"')
    print(f'  Expected:    {expected}')
    print(f'  BioBERT:     {t_label} (conf: {t_probs[t_label]:.3f})')
    print(f'  RNN/LSTM:    {r_label} (conf: {r_probs[r_label]:.3f})')

In [ ]:
# Visualize prediction probabilities for one example
example = "Patients treated with metformin showed a significant reduction in HbA1c levels (p<0.001)."

t_label, t_probs = predict_sentence(example, 'transformer')
r_label, r_probs = predict_sentence(example, 'rnn')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = list(t_probs.keys())

axes[0].barh(labels, [t_probs[l] for l in labels], color='coral')
axes[0].set_xlabel('Probability')
axes[0].set_title(f'BioBERT Prediction: {t_label}')
axes[0].set_xlim(0, 1)

axes[1].barh(labels, [r_probs[l] for l in labels], color='steelblue')
axes[1].set_xlabel('Probability')
axes[1].set_title(f'RNN/LSTM Prediction: {r_label}')
axes[1].set_xlim(0, 1)

plt.suptitle(f'Sentence: "{example[:70]}..."', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'inference_demo.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Error Analysis

In [ ]:
# Find misclassified examples from transformer model
test_texts = [dataset['test'][i]['text'] for i in range(len(dataset['test']))]

errors = []
for i, (true, pred) in enumerate(zip(trans_labels, trans_preds)):
    if true != pred:
        errors.append({
            'text': test_texts[i],
            'true_label': ID_TO_LABEL[true],
            'pred_label': ID_TO_LABEL[pred],
        })

print(f'Total misclassified: {len(errors)} / {len(trans_labels)} ({len(errors)/len(trans_labels)*100:.1f}%)')
print(f'\nSample misclassified sentences:')
for err in errors[:5]:
    print(f'\n  True: {err["true_label"]} -> Predicted: {err["pred_label"]}')
    print(f'  Text: {err["text"][:120]}...' if len(err['text']) > 120 else f'  Text: {err["text"]}')

In [ ]:
# Error distribution analysis
error_pairs = Counter([(e['true_label'], e['pred_label']) for e in errors])
print('Most common confusion pairs:')
for (true, pred), count in error_pairs.most_common(10):
    print(f'  {true} -> {pred}: {count}')

## Summary

This demo showed:
1. **Model Loading**: Both RNN/LSTM and BioBERT models can be loaded from saved checkpoints
2. **Test Evaluation**: Full test set evaluation with classification reports and confusion matrices
3. **Model Comparison**: BioBERT outperforms the RNN baseline across all metrics
4. **Interactive Inference**: Real-time predictions on custom biomedical sentences
5. **Error Analysis**: Common confusion patterns and misclassified examples